# Kaggle eval runner (thin)

Runs `src.evaluate` with vLLM on one T4. Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Optional secret: `HF_TOKEN` (Add-ons → Secrets).

The defaults below are the **Step 0 smoke test**: untrained Qwen2.5-3B-Instruct on dev, written to the ignored `results/raw/`. This checks the environment only; the base model is not a study condition. For a real eval, change `MODEL` / `RUN_NAME`. `--split test` is refused unless `--final` is passed.

Every shell command goes through `run()`, which raises on a nonzero exit, so **a failure stops the notebook at the cell that caused it** (use *Run All*).

In [ ]:
# --- run settings ---
REPO = "https://github.com/twiggytom/tiny-gambits.git"
COMMIT = "main"            # pin to a commit hash for reported runs
MODEL = "Qwen/Qwen2.5-3B-Instruct"
SPLIT = "dev"
RUN_NAME = "raw/smoke_3b_base"
GPU = "0"                  # one eval job per T4; use "1" for a second job in parallel
SEED = 0
LIMIT = None               # e.g. 20 for a quick check

import os, subprocess

def run(cmd: str) -> None:
    """Run a shell command, stream its output into the notebook, raise if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")

In [ ]:
run("nvidia-smi")

In [ ]:
os.chdir("/kaggle/working")
run(f"rm -rf tiny-gambits && git clone -q {REPO} tiny-gambits")
os.chdir("/kaggle/working/tiny-gambits")
run(f"git checkout -q {COMMIT} && git log -1 --format='%H %s'")

In [ ]:
# Kaggle ships pandas/numpy/pyarrow; install the pinned GPU eval set (requirements-gpu.txt) on top.
# Resolver warnings about unrelated preinstalled packages (gradio, litellm, cudf, ...) are expected.
run("pip install -q -r requirements-gpu.txt")
run("pip freeze > /kaggle/working/pip_freeze.txt")
run("grep -iE '^(torch|vllm|transformers|chess|pandas|numpy|pyarrow|huggingface.hub)==' /kaggle/working/pip_freeze.txt")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret (fine for public models/datasets):", type(e).__name__)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU

In [ ]:
limit = f"--limit {LIMIT}" if LIMIT else ""
run(f"python -m src.evaluate --config configs/eval.yaml --model {MODEL} --backend vllm --split {SPLIT} --run-name {RUN_NAME} --seed {SEED} {limit}")

In [ ]:
import json, shutil
out = f"results/{RUN_NAME}/{SPLIT}"
shutil.copy("/kaggle/working/pip_freeze.txt", f"{out}/pip_freeze.txt")
info = json.load(open(f"{out}/run_info.json"))
print(json.dumps({k: info[k] for k in ["gpu", "dtype", "seconds", "versions"]}, indent=2))
assert info["dtype"] == "float16" or "T4" not in info["gpu"], "T4 must run in fp16"
m = json.load(open(f"{out}/metrics.json"))
print(json.dumps({k: m[k] for k in ["n_puzzles", "first_move_acc", "full_solution_acc", "illegal_rate", "unparseable_rate"]}, indent=2))
with open(f"{out}/puzzles.jsonl") as f:
    for _ in range(3):
        r = json.loads(f.readline()); print(r["PuzzleId"], r["outcome"], r["model_outputs"])

In [ ]:
# vLLM vs HF parity check on the first 20 puzzles (same model, greedy). Outputs should match;
# rare fp16 kernel differences are possible and get logged, not hidden.
import pandas as pd
run(f"python -m src.evaluate --config configs/eval.yaml --model {MODEL} --backend hf --split {SPLIT} --limit 20 --run-name raw/parity_hf --seed {SEED}")
v = pd.read_json(f"{out}/puzzles.jsonl", lines=True, dtype={"PuzzleId": str}).set_index("PuzzleId")
h = pd.read_json(f"results/raw/parity_hf/{SPLIT}/puzzles.jsonl", lines=True, dtype={"PuzzleId": str}).set_index("PuzzleId")
cols = ["model_outputs", "outcome"]
diff = [pid for pid in h.index if any(str(v.loc[pid, c]) != str(h.loc[pid, c]) for c in cols)]
print(f"{len(h) - len(diff)}/{len(h)} puzzles identical; differing: {diff}")
for pid in diff:
    print(pid, "vllm:", v.loc[pid, "model_outputs"], "| hf:", h.loc[pid, "model_outputs"])

In [ ]:
# Zip results (main run + parity run) for download from the Output tab.
shutil.copytree("results/raw/parity_hf", f"{out}/parity_hf", dirs_exist_ok=True)
print(shutil.make_archive(f"/kaggle/working/{RUN_NAME.replace('/', '_')}_{SPLIT}", "zip", out))